# Tier 1: 250-Step Screen (~19 min)

**Purpose**: Fast kill for bad mutations. Fail fast, move on.

| Decision | Condition |
|----------|----------|
| ❌ Kill | step-250 BPB **> 2.10** |
| ⚠️ Marginal | step-250 BPB **2.00 – 2.10** |
| ✅ Promote to Tier 2 | step-250 BPB **< 2.00** |

**Baseline reference**: 2.0403 BPB @ step 250 (T4, 32k batch, seed 42)

In [ ]:
# [CELL 1] CLONE
%cd /content
!rm -rf uniform-int4
!git clone https://github.com/jmoncayo-pursuit/parameter-golf-uniform-int4.git uniform-int4
print('✅ Repo cloned')

In [ ]:
# [CELL 2] ASSETS
%cd /content/uniform-int4
!mkdir -p data/tokenizers data/datasets/fineweb10B_sp1024
!wget -q -O data/tokenizers/fineweb_1024_bpe.model \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/tokenizers/fineweb_1024_bpe.model
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_val_000000.bin
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_train_000001.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_train_000001.bin
!pip install -q sentencepiece zstandard PyYAML huggingface-hub datasets tqdm

import os
assert os.path.getsize('data/tokenizers/fineweb_1024_bpe.model') > 100_000
assert os.path.getsize('data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin') > 1_000_000
assert os.path.getsize('data/datasets/fineweb10B_sp1024/fineweb_train_000001.bin') > 100_000_000
print('✅ All assets verified')

In [ ]:
# [CELL 3] 250-STEP SCREEN
%cd /content/uniform-int4
!env NO_COMPILE=1 ITERATIONS=250 VAL_LOSS_EVERY=250 python3 train_gpt.py

In [ ]:
# [CELL 4] VERDICT
# Paste the step-250 val_bpb value here after the run completes:
bpb = float(input('Enter step-250 val_bpb: '))

BASELINE = 2.0403
if bpb > 2.10:
    print(f'❌ KILL — {bpb:.4f} is worse than baseline ({BASELINE}). Discard this mutation.')
elif bpb > BASELINE:
    print(f'⚠️ MARGINAL — {bpb:.4f} is close to baseline ({BASELINE}). Re-run with different seed before deciding.')
elif bpb > 2.00:
    print(f'🔶 PROMISING — {bpb:.4f} beats baseline ({BASELINE}). Consider Tier 2 proof (750 steps).')
else:
    print(f'✅ PROMOTE — {bpb:.4f} clearly beats baseline ({BASELINE}). Run Tier 2 proof immediately.')